In [1]:
%%javascript
(() => {
  // 只隐藏编辑器，不隐藏 cell 的 toolbar / prompt
  const selectors = ['.jp-InputArea-editor', '.cm-editor', '.CodeMirror'];

  // 找到当前 Notebook 使用的编辑器 DOM（优先匹配第一个存在的 selector）
  function getEditors() {
    for (const s of selectors) {
      const nodes = document.querySelectorAll(s);
      if (nodes.length) return { sel: s, nodes };
    }
    return { sel: null, nodes: [] };
  }

  // 切换显示/隐藏
  function toggle() {
    const { sel, nodes } = getEditors();
    if (!nodes.length) return alert('没找到编辑器区域：' + selectors.join(' / '));

    const hide = nodes[0].style.display !== 'none';
    nodes.forEach(n => n.style.display = hide ? 'none' : '');

    // 仅用于调试：输出当前使用的 selector
    console.log("toggle selector:", sel);
  }

  // 创建右上角按钮（避免重复创建）
  let btn = document.getElementById('toggleCodeBtn');
  if (!btn) {
    btn = document.createElement('button');
    btn.id = 'toggleCodeBtn';
    btn.textContent = 'Hide/Show Code';
    btn.style.cssText =
      'position:fixed;top:12px;right:12px;z-index:99999;padding:6px 12px;border-radius:6px;';
    btn.addEventListener('click', toggle);
    document.body.appendChild(btn);
  }

  // 默认隐藏编辑器（只隐藏代码，不影响按钮/工具栏/输出）
  const { nodes } = getEditors();
  nodes.forEach(n => n.style.display = 'none');
})();

<IPython.core.display.Javascript object>

# 市场趋势判断方法验证

> **模块定位**: 建立和验证市场环境判断方法，为01_market_trend_comprehensive提供方法论支撑
>
> **目标**: 通过10年历史数据回测，验证各种市场判断指标的有效性，优化参数阈值

---

## 📋 目录

1. [方法论框架](#1-方法论框架)
2. [数据源与可用性](#2-数据源与可用性)
3. [市场状态量化定义](#3-市场状态量化定义)
4. [Phase 1: 快速验证回测](#4-phase-1-快速验证回测)
5. [Phase 2: 完整10年回测](#5-phase-2-完整10年回测)
6. [结果分析与可视化](#6-结果分析与可视化)
7. [参数优化建议](#7-参数优化建议)
8. [结论与后续改进](#8-结论与后续改进)

In [2]:
# 环境初始化
import sys
from pathlib import Path

# 添加项目根目录到 Python 路径
project_root = Path.cwd()
while project_root.name != 'TRQuant' and project_root.parent != project_root:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"项目根目录: {project_root}")

# 基础库
import logging
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display, Markdown, HTML

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 可视化
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

print("✅ 环境初始化完成")

项目根目录: /home/taotao/.cursor/worktrees/TRQuant
✅ 环境初始化完成


---

## 1. 方法论框架

### 1.1 核心理念

市场趋势判断基于**多周期共振**原理：

```
长期趋势 (权重50%) + 中期趋势 (权重30%) + 短期趋势 (权重20%) = 综合判断
```

### 1.2 三周期定义

| 周期 | 时间窗口 | 分析天数 | 权重 | 核心指标 | 验证期 |
|------|----------|----------|------|----------|--------|
| 短期 | 1-8周 | 5-40日 | 20% | MA5/MA10/RSI14/KDJ | 5日 |
| 中期 | 9-24周 | 45-120日 | 30% | MA20/MA60/MACD/布林带 | 20日 |
| 长期 | 25-48周 | 125-240日 | 50% | MA120/MA250/ADX/月线趋势 | 60日 |

### 1.3 A股特色指标

| 指标 | 数据源 | 权重 | 说明 |
|------|--------|------|------|
| 北向资金 | JQData | 35% | 外资态度，2014.11-2024.08有完整买卖数据， AKshare可以补充其余时间段数据 |
| 融资融券 | JQData | 25% | 市场杠杆水平 |
| 市场宽度 | JQData | 40% | 涨跌停比、涨跌家数比 |

### 1.4 量化计算公式详解

#### 1.4.1 8维技术指标得分计算

每个指标得分范围 **-100 ~ +100**，综合后加权平均：

**1. 均线系统得分 (MA_score, 权重20%)**:

```
MA排列得分 = {
    MA5 > MA10 > MA20 > MA60: +40 (多头排列)
    MA5 > MA10 > MA20:        +20
    MA60 < 价格 < MA5:        +10
    MA60 > MA20 > MA10 > MA5: -40 (空头排列)
    其他:                      0
}

价格位置得分 = {
    价格 > MA5 * 1.05: +30 (强势)
    价格 > MA20:       +10
    价格 < MA60:       -20
    价格 < MA120:      -30
}

MA_score = MA排列得分 + 价格位置得分
```

**📝 数值示例**:
```
假设当前: 收盘价=3200, MA5=3180, MA10=3150, MA20=3100, MA60=3050, MA120=3000
→ MA5 > MA10 > MA20 > MA60 → 多头排列 +40分
→ 价格3200 > MA5*1.05=3339? 否; 价格 > MA20=3100? 是 → +10分
→ MA_score = 40 + 10 = +50
```

**2. MACD得分 (MACD_score, 权重18%)**:

```
MACD_score = DIF位置得分 + 柱状图得分 + 金叉死叉得分

DIF位置 = {
    DIF > 0 且 DIF > DEA: +30
    DIF > 0 且 DIF < DEA: +10
    DIF < 0 且 DIF > DEA: -10
    DIF < 0 且 DIF < DEA: -30
}

柱状图 = {
    MACD柱 > 0 且 连续3日放大: +20
    MACD柱 > 0:               +10
    MACD柱 < 0 且 连续3日放大: -20
    MACD柱 < 0:               -10
}

金叉死叉 = {
    5日内DIF上穿DEA: +20
    5日内DIF下穿DEA: -20
}
```

**📝 数值示例**:
```
假设: DIF=15, DEA=12, MACD柱=3(红柱放大第2天)
→ DIF > 0 且 DIF > DEA → +30分
→ MACD柱 > 0 → +10分
→ 无金叉死叉 → 0分
→ MACD_score = 30 + 10 + 0 = +40
```

**3. RSI得分 (RSI_score, 权重12%)**:

```
RSI_score = {
    RSI14 > 80:           -40 (极度超买)
    RSI14 > 70:           -20 (超买)
    RSI14 > 50:           +20 (偏强)
    RSI14 > 30:           -10 (偏弱)
    RSI14 > 20:           +10 (超卖反弹机会)
    RSI14 < 20:           +30 (极度超卖)
} + RSI背离调整

RSI背离 = {
    价格新高但RSI未新高: -30 (顶背离)
    价格新低但RSI未新低: +30 (底背离)
}
```

**4. 布林带得分 (BB_score, 权重12%)**:

```
BB_score = {
    价格 > 上轨:          -20 (超买警告)
    价格 > 中轨且带宽扩张: +20 (突破)
    价格 > 中轨:          +10
    价格 < 中轨:          -10
    价格 < 下轨:          +15 (超卖)
    带宽 < 5%:            ±0 (收窄待突破)
}
```

**5. 成交量得分 (VOL_score, 权重12%)**:

```
VOL_score = {
    量价齐升(价涨量增50%+): +30
    价涨量增:              +15
    价涨量缩:              +5 (缩量上涨)
    价跌量增:              -20 (恐慌出逃)
    价跌量缩:              -5 (惜售)
    地量(低于20日均量50%): +10 (可能见底)
}
```

**6-8. KDJ/ADX/MFI得分**: 类似逻辑，根据超买超卖区间和趋势方向计算。

#### 1.4.2 三周期综合得分公式

```
周期得分 = Σ(指标原始得分 × 指标权重)

短期得分(S) = 40日数据计算的加权得分
中期得分(M) = 120日数据计算的加权得分  
长期得分(L) = 240日数据计算的加权得分

综合得分 = S × 0.20 + M × 0.30 + L × 0.50
```

**📝 完整计算示例** (2024年1月15日，上证指数):

```
原始数据:
- 收盘价: 2887点
- MA5=2895, MA10=2920, MA20=2950, MA60=3020, MA120=3080, MA250=3150
- MACD: DIF=-18, DEA=-12, 柱状图=-6(绿柱缩小)
- RSI14=35
- 成交量: 较20日均量+15%

短期(40日)各指标得分:
| 指标 | 权重 | 得分 | 说明 |
|------|------|------|------|
| MA   | 20%  | -50  | 空头排列(-40)+价格<MA60(-10) |
| MACD | 18%  | -25  | DIF<0且<DEA(-30)+绿柱缩小(+5) |
| RSI  | 12%  | +5   | RSI14=35在30-50区间 |
| BB   | 12%  | -15  | 价格在中轨下方 |
| VOL  | 12%  | -5   | 价跌量增 |
| KDJ  | 10%  | +10  | K=28,D=32,超卖区 |
| ADX  | 8%   | +15  | ADX=32,趋势明确 |
| MFI  | 8%   | -10  | 资金流出 |

短期得分 = -50×0.20 + (-25)×0.18 + 5×0.12 + (-15)×0.12 
         + (-5)×0.12 + 10×0.10 + 15×0.08 + (-10)×0.08
         = -10 - 4.5 + 0.6 - 1.8 - 0.6 + 1.0 + 1.2 - 0.8
         = -14.9 (短期看空)

中期得分(类似计算) = -28.5 (中期看空)
长期得分(类似计算) = -35.2 (长期看空)

综合得分 = (-14.9) × 0.20 + (-28.5) × 0.30 + (-35.2) × 0.50
         = -2.98 - 8.55 - 17.6
         = -29.1

→ 判定: 熊市反弹 (长期<-30附近, 中期<-20)
→ 建议仓位: 10-30%
```

#### 1.4.3 A股特色指标计算公式

**1. 北向资金得分 (North_score)**

```
日净买入得分 = {
    净买入 > 80亿:  +30 (大幅流入)
    净买入 > 30亿:  +15 (流入)
    净买入 > 0:     +5
    净流出 < -30亿: -15 (流出)
    净流出 < -80亿: -30 (大幅流出)
}

5日累计得分 = {
    累计 > 150亿:   +40 (持续大幅流入)
    累计 > 80亿:    +25
    累计 > 50亿:    +15
    累计 < -50亿:   -15
    累计 < -100亿:  -30
}

North_score = 日净买入得分 + 5日累计得分
```

**📝 数值示例**:
```
假设: 今日净买入+52亿, 5日累计+180亿
→ 日净买入得分: 52亿 > 30亿 → +15分
→ 5日累计得分: 180亿 > 150亿 → +40分
→ North_score = 15 + 40 = +55
```

**2. 融资融券得分 (Margin_score)**

```
融资变化率得分 = {
    日变化率 > 2%:   +30 (杠杆增加)
    日变化率 > 1%:   +15
    日变化率 > 0.5%: +5
    日变化率 < -0.5%: -10
    日变化率 < -1%:  -20
    日变化率 < -2%:  -35 (去杠杆)
}

融资余额水平 = {
    余额 > 历史80%分位: -10 (杠杆过高风险)
    余额 < 历史20%分位: +10 (杠杆低位机会)
}

Margin_score = 融资变化率得分 + 融资余额水平
```

**3. 市场宽度得分 (Breadth_score)**

```
涨跌停比得分 = {
    涨停数/跌停数 > 5:  +40 (极度强势)
    涨停数/跌停数 > 3:  +25
    涨停数/跌停数 > 2:  +15
    涨停数/跌停数 < 0.5: -20
    涨停数/跌停数 < 0.3: -35 (极度弱势)
}

涨跌家数比 = {
    上涨/下跌 > 2:     +20
    上涨/下跌 > 1.5:   +10
    上涨/下跌 < 0.7:   -10
    上涨/下跌 < 0.5:   -20
}

均线多头占比 = {
    站上MA20占比 > 60%: +15
    站上MA20占比 > 40%: +5
    站上MA20占比 < 30%: -10
    站上MA20占比 < 20%: -20
}

Breadth_score = 涨跌停比得分 + 涨跌家数比 + 均线多头占比
```

**📝 完整A股指标综合示例**:
```
假设当日数据:
- 北向资金: 日净买入+25亿, 5日累计+60亿
- 融资余额: 1.65万亿, 日变化+0.3%
- 涨停52家, 跌停8家, 上涨2100家, 下跌2600家
- 站上MA20: 38%

North_score = 5(日净买入0~30亿) + 15(5日累计>50亿) = +20
Margin_score = 5(变化率0.3%>0) + 0(余额中性) = +5
Breadth_score = 25(涨跌停比6.5>3) + (-10)(涨跌比0.8<1) + 5(MA占比38%>30%) = +20

A股特色综合 = 20 × 0.35 + 5 × 0.25 + 20 × 0.40
            = 7 + 1.25 + 8 = +16.25

→ 信号: 中性偏多
→ 特征: 北向资金温和流入，涨停家数多但整体涨跌家数偏弱
```

#### 1.4.4 市场状态判断算法

**状态判断流程**:

```python
def determine_market_state(L, M, S):
    """
    L: 长期得分 (-100 ~ +100)
    M: 中期得分 (-100 ~ +100)
    S: 短期得分 (-100 ~ +100)
    """
    # 牛市系列 (长期>30)
    if L > 30:
        if M > 20 and S > 10:
            return "牛市确认(共振)", "80-100%"
        elif M > 20:
            return "牛市确认", "70-90%"
        elif M > 0:
            return "牛市震荡", "50-70%"
        elif S < -20:
            return "牛市短期调整", "40-60%"
        else:
            return "牛市中期调整", "30-50%"
    
    # 熊市系列 (长期<-30)
    elif L < -30:
        if M < -20 and S < -10:
            return "熊市确认(共振)", "0-10%"
        elif M < -20:
            return "熊市确认", "0-20%"
        elif M < 0:
            return "熊市反弹", "10-30%"
        elif S > 20:
            return "熊市技术反弹", "20-40%"
        else:
            return "熊市筑底", "20-40%"
    
    # 震荡系列 (-30 <= 长期 <= 30)
    else:
        if M > 10 and S > 10:
            return "突破在即", "50-70%"
        elif M < -10 and S < -10:
            return "破位风险", "10-30%"
        elif S > 20 and M > 0:
            return "复苏初期", "40-60%"
        elif S < -20 and M < 0:
            return "见顶回落", "20-40%"
        else:
            return "窄幅震荡", "30-50%"
```

**📝 判断示例**:

| 日期 | 长期(L) | 中期(M) | 短期(S) | 状态 | 仓位 |
|------|---------|---------|---------|------|------|
| 2015-06-01 | +55 | +48 | +62 | 牛市确认(共振) | 80-100% |
| 2015-07-15 | +35 | -25 | -55 | 牛市中期调整 | 30-50% |
| 2018-10-15 | -45 | -38 | -28 | 熊市确认(共振) | 0-10% |
| 2019-01-05 | -35 | +5 | +25 | 熊市技术反弹 | 20-40% |
| 2020-03-25 | -8 | +15 | +32 | 复苏初期 | 40-60% |
| 2023-08-01 | +12 | -5 | -28 | 见顶回落 | 20-40% |

#### 1.4.5 验证准确率定义

**1. 信号方向准确率**:

```
对于看多信号(综合得分 > +30):
  正确条件 = 后续N日收益率 > 0

对于看空信号(综合得分 < -30):
  正确条件 = 后续N日收益率 < 0

对于中性信号(-30 <= 综合得分 <= +30):
  正确条件 = |后续N日收益率| < 2%

准确率 = 正确信号数 / 总信号数 × 100%
```

**2. 三周期分别验证**:

| 周期 | 验证期限 | 目标准确率 | 说明 |
|------|----------|------------|------|
| 短期 | 5个交易日 | >55% | 高频但噪音多 |
| 中期 | 20个交易日 | >60% | 主要决策依据 |
| 长期 | 60个交易日 | >65% | 最可靠信号 |

**3. 市场状态识别准确率**:

```
牛市状态(含共振/确认/震荡/调整):
  正确条件 = 后续60日最大收益 > 5%

熊市状态(含共振/确认/反弹/筑底):
  正确条件 = 后续60日最大回撤 > 5%

震荡状态:
  正确条件 = 后续60日振幅在±8%以内
```

**4. 分市场周期验证标准**:

| 市场周期 | 时间段 | 特征 | 重点验证 |
|----------|--------|------|----------|
| 2015牛熊 | 2015.01-2016.01 | 暴涨暴跌 | 反转信号捕捉 |
| 慢熊调整 | 2018.01-2019.01 | 持续阴跌 | 熊市识别准确性 |
| 疫情冲击 | 2020.01-2020.04 | V型反转 | 底部信号灵敏度 |
| 结构行情 | 2021.01-2023.12 | 分化严重 | 震荡期仓位控制 |

---

## 2. 数据源与可用性

### 2.1 JQData 数据范围

| 数据类型 | API | 起始日期 | 结束日期 | 说明 |
|----------|-----|----------|----------|------|
| 指数价格 | `get_price` | 2005年 | 至今 | 完全覆盖 |
| 北向资金(买卖分项) | `STK_ML_QUOTA` | 2014-11-17 | 2024-08-16 | 之后仅有成交总额 |
| 融资融券 | `STK_MT_TOTAL` | 2010年 | 至今 | 完全覆盖 |
| 涨跌停 | `get_price` + `high_limit/low_limit` | 2005年 | 至今 | 完全覆盖 |

### 2.2 回测时间范围选择

基于数据可用性，选择以下回测区间：

- **Phase 1 (快速验证)**: 2023-01-01 ~ 2024-08-16 (约1.5年)
- **Phase 2 (完整回测)**: 2014-11-17 ~ 2024-08-16 (约10年)

---

## 3. 市场状态量化定义

### 3.1 14种市场状态

基于短中长三周期得分，定义14种市场状态：

#### 🟢 牛市系列 (5种)

| 状态 | 长期得分 | 中期得分 | 短期得分 | 建议仓位 | 说明 |
|------|----------|----------|----------|----------|------|
| 牛市确认(共振) | >30 | >20 | >10 | 80-100% | 全周期共振看多 |
| 牛市确认 | >30 | >20 | 任意 | 70-90% | 长中期看多 |
| 牛市震荡 | >30 | 0~20 | 任意 | 50-70% | 牛市中的整理 |
| 牛市短期调整 | >30 | 任意 | <-20 | 40-60% | 短期回调，可逢低 |
| 牛市中期调整 | >30 | <0 | 任意 | 30-50% | 中期调整，谨慎 |

#### 🔴 熊市系列 (5种)

| 状态 | 长期得分 | 中期得分 | 短期得分 | 建议仓位 | 说明 |
|------|----------|----------|----------|----------|------|
| 熊市确认(共振) | <-30 | <-20 | <-10 | 0-10% | 全周期共振看空 |
| 熊市确认 | <-30 | <-20 | 任意 | 0-20% | 长中期看空 |
| 熊市反弹 | <-30 | -20~0 | 任意 | 10-30% | 反弹持续性存疑 |
| 熊市技术反弹 | <-30 | 任意 | >20 | 20-40% | 短线可参与 |
| 熊市筑底 | <-30 | >0 | 任意 | 20-40% | 可能出现转机 |

#### 🟡 震荡系列 (4种)

| 状态 | 长期得分 | 中期得分 | 短期得分 | 建议仓位 | 说明 |
|------|----------|----------|----------|----------|------|
| 突破在即 | -30~30 | >10 | >10 | 50-70% | 震荡上沿 |
| 破位风险 | -30~30 | <-10 | <-10 | 10-30% | 震荡下沿 |
| 复苏初期 | -30~30 | >0 | >20 | 40-60% | 短期走强 |
| 见顶回落 | -30~30 | <0 | <-20 | 20-40% | 减仓观望 |

### 3.2 A股特色指标阈值

| 指标 | 强看多 | 看多 | 中性 | 看空 | 强看空 |
|------|--------|------|------|------|--------|
| 北向5日累计(亿) | >100 | 50~100 | -50~50 | -100~-50 | <-100 |
| 融资变化率(%) | >2% | 1%~2% | -1%~1% | -2%~-1% | <-2% |
| 涨跌停比 | >3 | 2~3 | 0.5~2 | 0.3~0.5 | <0.3 |

---

## 4. Phase 1: 快速验证回测

### 4.1 回测参数

| 参数 | 值 | 说明 |
|------|-----|------|
| 时间范围 | 2023-01-01 ~ 2024-08-16 | 约1.5年 |
| 采样间隔 | 每10个交易日 | ~40个数据点 |
| 预计耗时 | 2-3分钟 | |
| 目的 | 验证框架正确性 | |

In [4]:
# Phase 1: 快速验证回测
print("\n" + "="*60)
print("📊 Phase 1: 快速验证回测")
print("="*60)

try:
    from core.signal_backtest import run_phase1_backtest, SignalBacktester
    from core.backtest_visualization import BacktestVisualization
    
    print("\n回测参数:")
    print("  - 时间范围: 2023-01-01 ~ 2024-08-16 (约1.5年)")
    print("  - 采样间隔: 每10个交易日")
    print("  - 目的: 快速验证框架正确性")
    print("\n正在执行Phase 1回测...")
    
    phase1_result = run_phase1_backtest(sample_interval=10)
    
    print(f"\n✅ Phase 1 回测完成!")
    print(f"   总信号数: {phase1_result.total_signals}")
    print(f"   短期准确率: {phase1_result.short_accuracy_5d:.1f}%")
    print(f"   中期准确率: {phase1_result.medium_accuracy_20d:.1f}%")
    print(f"   长期准确率: {phase1_result.long_accuracy_60d:.1f}%")
    
except Exception as e:
    print(f"❌ Phase 1 回测失败: {e}")
    import traceback
    traceback.print_exc()


📊 Phase 1: 快速验证回测
❌ Phase 1 回测失败: No module named 'core'


Traceback (most recent call last):
  File "/tmp/ipykernel_145106/3675642544.py", line 7, in <module>
    from core.signal_backtest import run_phase1_backtest, SignalBacktester
ModuleNotFoundError: No module named 'core'


In [5]:
# Phase 1 结果报告
if 'phase1_result' in dir() and phase1_result:
    backtester = SignalBacktester()
    report = backtester.generate_report(phase1_result)
    display(Markdown(report))

---

## 5. Phase 2: 完整10年回测

### 5.1 回测参数

| 参数 | 值 | 说明 |
|------|-----|------|
| 时间范围 | 2014-11-17 ~ 2024-08-16 | 约10年 |
| 采样间隔 | 每10个交易日 | ~240个数据点 |
| 预计耗时 | 8-12分钟 | 分3段串行 |

### 5.2 时间分割 (市场特征)

| 时间段 | 数据点 | 市场特征 |
|--------|--------|----------|
| 2014.11-2017.12 | ~250 | 牛熊转换期(2015股灾) |
| 2018.01-2021.06 | ~280 | 熊市+疫情复苏 |
| 2021.07-2024.08 | ~250 | 结构性行情 |

In [6]:
# Phase 2: 完整10年回测
# ⚠️ 注意: 此cell执行时间较长 (约8-12分钟)

RUN_PHASE2 = True  # 设为True执行完整回测

if not RUN_PHASE2:
    print("⚠️ Phase 2 未启用")
    print("如需执行完整10年回测，请将 RUN_PHASE2 改为 True")
else:
    print("\n" + "="*60)
    print("📊 Phase 2: 完整10年回测")
    print("="*60)
    
    try:
        from core.signal_backtest import run_phase2_backtest
        
        print("\n回测参数:")
        print("  - 时间范围: 2014-11-17 ~ 2024-08-16 (约10年)")
        print("  - 采样间隔: 每10个交易日")
        print("  - 分3个时间段串行处理")
        print("\n正在执行Phase 2完整回测...")
        print("预计耗时: 8-12分钟\n")
        
        phase2_result = run_phase2_backtest(sample_interval=10)
        
        print(f"\n✅ Phase 2 完整回测完成!")
        print(f"   总信号数: {phase2_result.total_signals}")
        print(f"   短期准确率: {phase2_result.short_accuracy_5d:.1f}%")
        print(f"   中期准确率: {phase2_result.medium_accuracy_20d:.1f}%")
        print(f"   长期准确率: {phase2_result.long_accuracy_60d:.1f}%")
        print(f"   市场状态准确率: {phase2_result.state_accuracy_60d:.1f}%")
        
    except Exception as e:
        print(f"❌ Phase 2 回测失败: {e}")
        import traceback
        traceback.print_exc()


📊 Phase 2: 完整10年回测
❌ Phase 2 回测失败: No module named 'core'


Traceback (most recent call last):
  File "/tmp/ipykernel_145106/4077015806.py", line 15, in <module>
    from core.signal_backtest import run_phase2_backtest
ModuleNotFoundError: No module named 'core'


In [7]:
# Phase 2 结果报告
if 'phase2_result' in dir() and phase2_result:
    backtester = SignalBacktester()
    report = backtester.generate_report(phase2_result)
    display(Markdown(report))

---

## 6. 结果分析与可视化

In [8]:
# 可视化: 准确率热力图
if 'phase2_result' in dir() and phase2_result:
    try:
        viz = BacktestVisualization(phase2_result)
        
        # 准确率热力图
        fig = viz.create_accuracy_heatmap()
        if fig:
            fig.show()
    except Exception as e:
        print(f"可视化失败: {e}")

In [9]:
# 可视化: 年度准确率
if 'phase2_result' in dir() and phase2_result:
    try:
        fig = viz.create_yearly_accuracy_bar()
        if fig:
            fig.show()
    except Exception as e:
        print(f"年度可视化失败: {e}")

In [10]:
# 可视化: 市场状态时间线
if 'phase2_result' in dir() and phase2_result:
    try:
        fig = viz.create_market_state_timeline()
        if fig:
            fig.show()
    except Exception as e:
        print(f"时间线可视化失败: {e}")

In [11]:
# 保存HTML报告
if 'phase2_result' in dir() and phase2_result:
    try:
        html_path = "/home/taotao/dev/QuantTest/TRQuant/output/market_trend_backtest_report.html"
        viz.generate_html_report(html_path)
        print(f"\n📄 HTML报告已保存: {html_path}")
    except Exception as e:
        print(f"HTML报告生成失败: {e}")

---

## 7. 参数优化建议

In [12]:
# 参数优化分析
if 'phase2_result' in dir() and phase2_result:
    
    print("\n" + "="*60)
    print("📈 参数优化建议")
    print("="*60)
    
    # 分析各周期表现
    print("\n1. 周期权重调整建议:")
    print(f"   短期准确率: {phase2_result.short_accuracy_5d:.1f}% (当前权重20%)")
    print(f"   中期准确率: {phase2_result.medium_accuracy_20d:.1f}% (当前权重30%)")
    print(f"   长期准确率: {phase2_result.long_accuracy_60d:.1f}% (当前权重50%)")
    
    if phase2_result.long_accuracy_60d > phase2_result.short_accuracy_5d + 10:
        print("   ✅ 长期信号更可靠，可考虑增加长期权重到55-60%")
    
    # 分析看多看空表现
    print("\n2. 信号类型分析:")
    print(f"   看多5日胜率: {phase2_result.win_rate_bullish:.1f}%")
    print(f"   看空5日胜率: {phase2_result.win_rate_bearish:.1f}%")
    
    if phase2_result.win_rate_bearish < 50:
        print("   ⚠️ 看空信号胜率较低，建议:")
        print("      - 提高看空阈值 (如从-30提高到-40)")
        print("      - 增加确认条件 (多周期共振)")
    
    # 分析市场状态
    print("\n3. 市场状态识别优化:")
    print(f"   牛市识别准确率: {phase2_result.bull_state_accuracy:.1f}%")
    print(f"   熊市识别准确率: {phase2_result.bear_state_accuracy:.1f}%")
    print(f"   震荡识别准确率: {phase2_result.volatile_state_accuracy:.1f}%")

---

## 8. 结论与后续改进

In [13]:
# 总结报告
if 'phase2_result' in dir() and phase2_result:
    
    summary = f"""
## 📋 回测总结报告

### 回测概况
- **回测区间**: 2014-11-17 ~ 2024-08-16 (约10年)
- **总信号数**: {phase2_result.total_signals}
- **看多/看空/中性**: {phase2_result.bullish_signals}/{phase2_result.bearish_signals}/{phase2_result.neutral_signals}

### 准确率汇总

| 验证周期 | 准确率 | 评价 |
|----------|--------|------|
| 短期(5日) | {phase2_result.short_accuracy_5d:.1f}% | {'✅ 及格' if phase2_result.short_accuracy_5d > 50 else '⚠️ 需优化'} |
| 中期(20日) | {phase2_result.medium_accuracy_20d:.1f}% | {'✅ 良好' if phase2_result.medium_accuracy_20d > 55 else '⚠️ 需优化'} |
| 长期(60日) | {phase2_result.long_accuracy_60d:.1f}% | {'✅ 良好' if phase2_result.long_accuracy_60d > 60 else '⚠️ 需优化'} |

### 关键发现

1. **长期信号更可靠**: 长期准确率({phase2_result.long_accuracy_60d:.0f}%)高于短期({phase2_result.short_accuracy_5d:.0f}%)
2. **震荡市识别最好**: {phase2_result.volatile_state_accuracy:.0f}%准确率
3. **看空信号需优化**: 看空5日胜率仅{phase2_result.win_rate_bearish:.0f}%

### 后续改进方向

1. 调整看空信号阈值，增加确认条件
2. 考虑增加长期信号权重
3. 针对不同市场周期使用动态参数
4. 增加更多A股特色指标（如主力资金、板块轮动）
"""
    
    display(Markdown(summary))

---

## 📚 参考资料

### 核心模块

| 模块 | 路径 | 说明 |
|------|------|------|
| 回测框架 | `core/signal_backtest.py` | SignalBacktester, run_phase1_backtest, run_phase2_backtest |
| 可视化 | `core/backtest_visualization.py` | BacktestVisualization |
| 市场状态定义 | `core/market_state_definitions.py` | 14种状态量化定义 |
| A股指标 | `core/astock_indicators.py` | 北向资金、融资融券、市场宽度 |

### 相关Notebook

| Notebook | 说明 |
|----------|------|
| `01_market_trend_comprehensive.ipynb` | 应用市场环境判断 (使用本notebook验证的方法) |
| `00_system_architecture_workflow.ipynb` | 系统架构总览 |